#  Entrenamiento Clasificador LSM
**Proyecto:** Traductor de Lengua de Señas Mexicana
**Alumno:** Axel Lopez | CECyTEN Tepic | Módulo III IA

Este notebook entrena un clasificador de Machine Learning con los ángulos capturados de cada seña.


In [ ]:
# ── CELDA 1: Instalar librerías ──────────────────────────────
!pip install scikit-learn pandas numpy matplotlib seaborn joblib -q
print('✅ Librerías listas')

In [ ]:
# ── CELDA 2: Subir el dataset ────────────────────────────────
from google.colab import files
print('📂 Sube tu archivo dataset_señas.csv')
uploaded = files.upload()
print('✅ Archivo cargado:', list(uploaded.keys()))

In [ ]:
# ── CELDA 3: Cargar y explorar datos ────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('dataset_señas.csv')
print('Shape:', df.shape)
print('\nMuestras por letra:')
print(df['letra'].value_counts().sort_index())

# Gráfica de distribución
plt.figure(figsize=(12,4))
df['letra'].value_counts().sort_index().plot(kind='bar', color='steelblue')
plt.title('Muestras por letra')
plt.xlabel('Letra')
plt.ylabel('Cantidad')
plt.tight_layout()
plt.show()
print('✅ Dataset cargado correctamente')

In [ ]:
# ── CELDA 4: Preparar datos para entrenamiento ───────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

X = df[['angle1','angle2','angle3','angle4','angle5','angle6']].values
y = df['letra'].values

# Codificar etiquetas
le = LabelEncoder()
y_enc = le.fit_transform(y)

# Dividir en entrenamiento y prueba (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

print(f'Entrenamiento: {X_train.shape[0]} muestras')
print(f'Prueba:        {X_test.shape[0]} muestras')
print(f'Clases: {le.classes_}')

In [ ]:
# ── CELDA 5: Entrenar modelos y comparar ─────────────────────
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

modelos = {
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'KNN':                 KNeighborsClassifier(n_neighbors=5),
    'Árbol de Decisión':   DecisionTreeClassifier(random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
}

resultados = {}
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    acc = accuracy_score(y_test, modelo.predict(X_test))
    resultados[nombre] = acc
    print(f'{nombre:25s}: {acc*100:.2f}%')

mejor_nombre = max(resultados, key=resultados.get)
print(f'\n🏆 Mejor modelo: {mejor_nombre} ({resultados[mejor_nombre]*100:.2f}%)')

In [ ]:
# ── CELDA 6: Matriz de confusión del mejor modelo ────────────
from sklearn.metrics import confusion_matrix, classification_report

mejor_modelo = modelos[mejor_nombre]
y_pred = mejor_modelo.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(12,10))
sns.heatmap(cm, annot=True, fmt='d', 
            xticklabels=le.classes_, 
            yticklabels=le.classes_,
            cmap='Blues')
plt.title(f'Matriz de Confusión — {mejor_nombre}')
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.tight_layout()
plt.show()

print('\nReporte por letra:')
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# ── CELDA 7: Guardar modelo y descargarlo ───────────────────
import joblib

# Guardar modelo
joblib.dump(mejor_modelo, 'modelo_lsm.pkl')
# Guardar codificador de etiquetas
joblib.dump(le, 'label_encoder.pkl')

print(f'✅ Modelo guardado: modelo_lsm.pkl')
print(f'✅ Encoder guardado: label_encoder.pkl')
print(f'Precisión final: {resultados[mejor_nombre]*100:.2f}%')

# Descargar archivos
files.download('modelo_lsm.pkl')
files.download('label_encoder.pkl')